# De Redes Neuronales al Transformer
### Sesión Puente — Proyecto I: Introducción a LLMs
**Facultad de Ciencias, UNAM — Semestre 2026-2**

---

> **Objetivo de esta sesión:** Ya leyeron *Attention is All You Need*. Este notebook construye el piso que el paper asume que tienes. Al final, cada ecuación del paper va a tener sentido.

**Mapa de la sesión:**

```
Parte 1: ¿Qué hace una neurona?        (~20 min)
Parte 2: Una red multicapa             (~20 min)
Parte 3: ¿Cómo aprende? Loss y gradiente (~20 min)
Parte 4: De la NN al Transformer       (~30 min)
Parte 5: Ejercicio de cierre           (~20 min)
```

**No necesitas GPU.** Todo corre en CPU en menos de 30 segundos.

In [ ]:
# Instalación (solo si es necesario)
# !pip install torch matplotlib numpy --quiet

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display

# Reproducibilidad
torch.manual_seed(42)
np.random.seed(42)

print("Todo listo. PyTorch version:", torch.__version__)

---
## Parte 1: ¿Qué hace una neurona?

Una neurona artificial hace exactamente tres cosas:

1. **Toma entradas** $x_1, x_2, \ldots, x_n$
2. **Las combina linealmente** con pesos: $z = w_1 x_1 + w_2 x_2 + \cdots + w_n x_n + b$
3. **Aplica una función no lineal** (activación): $a = \sigma(z)$

En notación vectorial:
$$a = \sigma(\mathbf{w}^\top \mathbf{x} + b)$$

Esto ya lo conocen: es exactamente una **regresión logística** cuando $\sigma$ = sigmoide.

### ¿Por qué la función de activación?

Sin activación, apilar capas lineales sigue siendo lineal:
$$W_2(W_1 x + b_1) + b_2 = (W_2 W_1)x + (W_2 b_1 + b_2) = W' x + b'$$

La no-linealidad es lo que hace que las redes puedan aprender funciones complejas.

In [ ]:
# Las activaciones más comunes
z = torch.linspace(-4, 4, 200)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Funciones de Activación', fontsize=14, fontweight='bold')

activaciones = [
    ('Sigmoide\n$\\sigma(z) = 1/(1+e^{-z})$', torch.sigmoid(z), '#2196F3'),
    ('ReLU\n$\\max(0, z)$', F.relu(z), '#4CAF50'),
    ('GELU\n(usada en GPT/BERT)', F.gelu(z), '#FF5722'),
]

for ax, (titulo, salida, color) in zip(axes, activaciones):
    ax.plot(z.numpy(), salida.numpy(), color=color, linewidth=2.5)
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
    ax.axvline(0, color='gray', linewidth=0.8, linestyle='--')
    ax.set_title(titulo, fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-4, 4)

plt.tight_layout()
plt.show()

print("Nota: GPT-2 usa GELU. BERT usa GELU. El Transformer original usa ReLU en el FFN.")
print("La elección de activación importa, pero el concepto es siempre el mismo.")

In [ ]:
# Una neurona a mano — sin nn.Module todavía
print("=" * 50)
print("Una neurona: clasificar si un texto es 'positivo'")
print("=" * 50)

# Supongamos que cada texto se representa con 3 features:
# [conteo_palabras_positivas, conteo_palabras_negativas, longitud_normalizada]

x = torch.tensor([3.0, 1.0, 0.5])  # ejemplo de un texto
w = torch.tensor([0.8, -0.6, 0.1]) # pesos (aprenderemos cómo se obtienen)
b = torch.tensor(-0.2)              # bias

# Paso 1: combinación lineal
z = torch.dot(w, x) + b
print(f"\nEntrada x = {x.tolist()}")
print(f"Pesos   w = {w.tolist()}")
print(f"Bias    b = {b.item():.1f}")
print(f"\nz = w·x + b = {z.item():.4f}")

# Paso 2: activación sigmoide
a = torch.sigmoid(z)
print(f"a = sigmoid(z) = {a.item():.4f}")
print(f"\nInterpretación: probabilidad de 'positivo' = {a.item()*100:.1f}%")

# Cambiar los pesos: ¿qué pasa?
print("\n--- ¿Qué pasa si cambiamos los pesos? ---")
for w_pos in [0.1, 0.5, 1.5, 3.0]:
    w_test = torch.tensor([w_pos, -0.6, 0.1])
    z_test = torch.dot(w_test, x) + b
    a_test = torch.sigmoid(z_test)
    print(f"w[0]={w_pos:.1f} → prob_positivo = {a_test.item()*100:.1f}%")

---
## Parte 2: Una Red Neuronal Multicapa (MLP)

Una red es simplemente **neuronas apiladas en capas**.

Para una red con una capa oculta:

$$\mathbf{h} = \sigma(W_1 \mathbf{x} + \mathbf{b}_1)$$
$$\hat{y} = \text{softmax}(W_2 \mathbf{h} + \mathbf{b}_2)$$

donde:
- $\mathbf{x} \in \mathbb{R}^d$ — vector de entrada
- $W_1 \in \mathbb{R}^{h \times d}$ — primera matriz de pesos
- $\mathbf{h} \in \mathbb{R}^h$ — activaciones de la capa oculta
- $W_2 \in \mathbb{R}^{|V| \times h}$ — segunda matriz de pesos
- $\hat{y} \in \mathbb{R}^{|V|}$ — distribución de probabilidad sobre clases

**Este bloque lineal → activación → lineal es exactamente el Feed-Forward Network (FFN) del Transformer.** Lo veremos en la Parte 4.

In [ ]:
# Construir una MLP con PyTorch

class MLP_Simple(nn.Module):
    def __init__(self, dim_entrada, dim_oculta, num_clases):
        super().__init__()
        # nn.Linear implementa: y = xW^T + b
        self.capa1 = nn.Linear(dim_entrada, dim_oculta)
        self.capa2 = nn.Linear(dim_oculta, num_clases)
    
    def forward(self, x):
        # Forward pass: la dirección hacia adelante
        h = F.relu(self.capa1(x))   # capa oculta con ReLU
        logits = self.capa2(h)       # capa de salida (sin activación — la loss la aplica)
        return logits


# Ejemplo: clasificar intención de texto
# Input: vector de 8 features (frecuencia de tokens)
# Clases: [pregunta, afirmación, negación]

modelo = MLP_Simple(dim_entrada=8, dim_oculta=16, num_clases=3)

print("Arquitectura de la red:")
print(modelo)
print()

# Contar parámetros
total_params = sum(p.numel() for p in modelo.parameters())
print(f"Total de parámetros: {total_params}")
print("Desglose:")
for nombre, param in modelo.named_parameters():
    print(f"  {nombre}: {param.shape} = {param.numel()} parámetros")

In [ ]:
# El Forward Pass
print("=" * 50)
print("Forward Pass: de input a predicción")
print("=" * 50)

# Simular un batch de 4 textos (batch_size=4, features=8)
x_batch = torch.randn(4, 8)  # 4 textos, 8 features cada uno
print(f"\nEntrada: {x_batch.shape}  (batch_size=4, features=8)")

# Paso por paso
with torch.no_grad():  # no necesitamos gradientes aquí
    # Capa 1
    z1 = modelo.capa1(x_batch)
    h = F.relu(z1)
    print(f"Después de capa 1 + ReLU: {h.shape}  (batch=4, oculta=16)")
    
    # Capa 2
    logits = modelo.capa2(h)
    print(f"Logits (salida bruta): {logits.shape}  (batch=4, clases=3)")
    
    # Convertir a probabilidades
    probs = F.softmax(logits, dim=-1)
    print(f"Probabilidades: {probs.shape}")
    
    # Predicción final
    predicciones = probs.argmax(dim=-1)
    nombres_clases = ["pregunta", "afirmación", "negación"]
    
    print("\nResultados para los 4 textos:")
    for i in range(4):
        p = probs[i]
        pred = nombres_clases[predicciones[i].item()]
        print(f"  Texto {i+1}: [{p[0]:.2f}, {p[1]:.2f}, {p[2]:.2f}] → {pred}")

print("\nImportante: estos pesos son ALEATORIOS — todavía no aprendió nada.")

### ¿Qué son los logits?

Los **logits** son los valores crudos antes de la softmax. Son scores no normalizados. La softmax los convierte en probabilidades:

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

**Conexión con el paper:** La Ecuación (1) de *Attention is All You Need* usa exactamente softmax:

$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

El softmax ahí convierte los scores de atención en **pesos de atención** (distribución de probabilidad que suma 1).

In [ ]:
# Intuición del softmax
print("Softmax: scores → probabilidades\n")

casos = [
    ([1.0, 1.0, 1.0], "Scores iguales → distribución uniforme"),
    ([3.0, 1.0, 0.5], "Un score dominante → concentración"),
    ([10.0, 1.0, 0.5], "Score muy alto → casi todo el peso ahí (sharp attention)"),
    ([0.1, 0.1, 0.1], "Scores pequeños pero iguales → uniforme de nuevo"),
]

for scores, descripcion in casos:
    z = torch.tensor(scores)
    p = F.softmax(z, dim=0)
    print(f"Scores: {scores}")
    print(f"  → {descripcion}")
    print(f"  → probs: [{p[0]:.3f}, {p[1]:.3f}, {p[2]:.3f}]  (suma={p.sum():.3f})")
    print()

print("Pregunta para reflexionar:")
print("En el mecanismo de atención, ¿qué significa que una probabilidad sea alta?")

---
## Parte 3: ¿Cómo Aprende? Loss, Gradiente, Backprop

El aprendizaje tiene tres pasos:

1. **Calcular el error** — función de pérdida (loss)
2. **Calcular el gradiente** — ¿en qué dirección mover los pesos?
3. **Actualizar los pesos** — descenso de gradiente

### La función de pérdida

Para clasificación se usa **Cross-Entropy Loss**:

$$\mathcal{L} = -\sum_i y_i \log(\hat{y}_i)$$

donde $y_i$ es la clase verdadera (one-hot) y $\hat{y}_i$ es la probabilidad predicha.

**Intuición:** Penaliza mucho cuando el modelo asigna probabilidad baja a la clase correcta.

### El gradiente

$$\frac{\partial \mathcal{L}}{\partial W} = \text{¿cuánto cambió la loss cuando cambiamos } W?$$

PyTorch calcula esto automáticamente con **autograd**. Nosotros solo definimos el forward pass.

In [ ]:
# Entrenamiento completo en un dataset de juguete
# Tarea: clasificar oraciones por sentimiento (pos/neg/neutro)

# Dataset sintético: vectores de 4 features
# Feature 0: proporción de palabras positivas
# Feature 1: proporción de palabras negativas
# Feature 2: intensidad (signos de exclamación, mayúsculas)
# Feature 3: longitud normalizada

X_train = torch.tensor([
    [0.8, 0.1, 0.7, 0.4],  # positivo
    [0.9, 0.0, 0.9, 0.6],  # positivo
    [0.1, 0.8, 0.5, 0.3],  # negativo
    [0.0, 0.9, 0.8, 0.7],  # negativo
    [0.4, 0.4, 0.1, 0.5],  # neutro
    [0.3, 0.3, 0.2, 0.4],  # neutro
    [0.7, 0.2, 0.6, 0.3],  # positivo
    [0.2, 0.7, 0.4, 0.6],  # negativo
])
y_train = torch.tensor([0, 0, 1, 1, 2, 2, 0, 1])  # 0=pos, 1=neg, 2=neutro

# Modelo
red = MLP_Simple(dim_entrada=4, dim_oculta=8, num_clases=3)

# Optimizador: SGD actualiza w ← w - lr * ∂L/∂w
optimizador = torch.optim.SGD(red.parameters(), lr=0.05)
criterio = nn.CrossEntropyLoss()

# Entrenamiento
historial_loss = []
historial_acc = []

for epoch in range(300):
    # 1. Forward pass
    logits = red(X_train)
    
    # 2. Calcular loss
    loss = criterio(logits, y_train)
    
    # 3. Backward pass (calcular gradientes)
    optimizador.zero_grad()  # limpiar gradientes anteriores
    loss.backward()          # backpropagation
    
    # 4. Actualizar pesos
    optimizador.step()
    
    # Registrar métricas
    historial_loss.append(loss.item())
    with torch.no_grad():
        preds = logits.argmax(dim=-1)
        acc = (preds == y_train).float().mean().item()
        historial_acc.append(acc)

print(f"Loss inicial:  {historial_loss[0]:.4f}")
print(f"Loss final:    {historial_loss[-1]:.4f}")
print(f"Accuracy final: {historial_acc[-1]*100:.1f}%")

In [ ]:
# Visualizar el entrenamiento
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(historial_loss, color='#E53935', linewidth=2)
ax1.set_title('Loss durante entrenamiento', fontsize=12, fontweight='bold')
ax1.set_xlabel('Época')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.grid(True, alpha=0.3)

ax2.plot([a*100 for a in historial_acc], color='#1E88E5', linewidth=2)
ax2.set_title('Accuracy durante entrenamiento', fontsize=12, fontweight='bold')
ax2.set_xlabel('Época')
ax2.set_ylabel('Accuracy (%)')
ax2.set_ylim(0, 105)
ax2.axhline(100, color='green', linestyle='--', alpha=0.5, label='100%')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("La loss bajó, la accuracy subió. Los pesos se ajustaron solos.")
print("Eso es el aprendizaje.")

In [ ]:
# Intuición del gradiente: un ejemplo simple
print("Intuición del gradiente")
print("=" * 40)

# Un solo parámetro: w
# La loss es: L(w) = (w - 3)^2  → mínimo en w=3

w = torch.tensor([-2.0], requires_grad=True)  # empezar lejos del mínimo
lr = 0.1

print(f"{'Paso':>5} | {'w':>8} | {'Loss':>10} | {'Gradiente':>12}")
print("-" * 45)

for paso in range(8):
    # Forward
    loss = (w - 3.0) ** 2
    
    # Backward
    loss.backward()
    
    print(f"{paso:>5} | {w.item():>8.4f} | {loss.item():>10.4f} | {w.grad.item():>12.4f}")
    
    # Actualizar: w = w - lr * gradiente
    with torch.no_grad():
        w -= lr * w.grad
        w.grad.zero_()

print(f"\nMínimo encontrado: w = {w.item():.4f}  (esperado: 3.0)")
print("\nObservar: el gradiente es positivo cuando w<3, negativo cuando w>3")
print("→ Siempre empuja w hacia el mínimo")

---
## Parte 4: De la Red Neuronal al Transformer

Ahora que tenemos el piso, vamos a ver qué hay exactamente en el Transformer. Cada bloque que vamos a implementar aparece en el paper.

### La arquitectura en una imagen

```
TRANSFORMER BLOCK (se repite N=6 veces en el paper)
┌─────────────────────────────────────────┐
│  Input: secuencia de vectores           │
│         [x₁, x₂, ..., xₙ]              │
│                   │                     │
│  ┌────────────────▼──────────────────┐  │
│  │   Multi-Head Self-Attention        │  │
│  │   (Lo que hace especial al        │  │
│  │    Transformer)                    │  │
│  └────────────────┬──────────────────┘  │
│                   │                     │
│              Add & LayerNorm            │
│                   │                     │
│  ┌────────────────▼──────────────────┐  │
│  │   Feed-Forward Network (FFN)       │  │
│  │   = Linear → ReLU → Linear        │  │
│  │   (¡Exactamente la MLP de Pt. 2!) │  │
│  └────────────────┬──────────────────┘  │
│                   │                     │
│              Add & LayerNorm            │
│                   │                     │
│  Output: secuencia transformada         │
└─────────────────────────────────────────┘
```

**Clave:** El FFN es idéntico para cada posición pero independiente. Dos tercios del bloque ya lo conocemos.

In [ ]:
# El Feed-Forward Network del Transformer
# Paper, Sección 3.3: "FFN(x) = max(0, xW₁ + b₁)W₂ + b₂"
# dim_modelo=512, dim_interna=2048 en el paper

class FeedForwardNetwork(nn.Module):
    """Exactamente el FFN de la Sección 3.3 del paper."""
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)   # xW₁ + b₁
        self.linear2 = nn.Linear(d_ff, d_model)   # ...W₂ + b₂
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # max(0, xW₁+b₁) es ReLU — igual que nuestra MLP_Simple
        return self.linear2(self.dropout(F.relu(self.linear1(x))))


# Probar con dimensiones del paper
d_model = 512   # dimensión del modelo
d_ff = 2048     # dimensión interna del FFN (×4)
seq_len = 10    # longitud de la secuencia
batch = 2

ffn = FeedForwardNetwork(d_model, d_ff)

# El input es una SECUENCIA de vectores (no un solo vector)
x = torch.randn(batch, seq_len, d_model)
salida = ffn(x)

print("Feed-Forward Network del Transformer")
print(f"Input:  {x.shape}  (batch, seq_len, d_model)")
print(f"Output: {salida.shape}  (batch, seq_len, d_model)  ← misma forma")
print()
print("El FFN se aplica independientemente a cada posición:")
print("→ token[0] se procesa igual que token[1], etc.")
print("→ NO hay interacción entre posiciones aquí (eso lo hace la atención)")

# Parámetros
params_ffn = sum(p.numel() for p in ffn.parameters())
print(f"\nParámetros del FFN: {params_ffn:,}")

### El Mecanismo de Atención

Ahora lo más importante. La fórmula del paper:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

¿Qué son Q, K, V?

Son tres transformaciones lineales del mismo input $x$:

$$Q = xW^Q, \quad K = xW^K, \quad V = xW^V$$

**Analogía:** Imagina una biblioteca.
- **Query (Q):** Tu pregunta de búsqueda — *"¿qué estoy buscando?"*
- **Key (K):** El índice de cada libro — *"¿de qué trata este libro?"*
- **Value (V):** El contenido de cada libro — *"esto es lo que puedes leer"*

El dot product $QK^\top$ mide qué tan relevante es cada libro para tu pregunta. El softmax convierte eso en pesos. El resultado es una mezcla ponderada del contenido.

In [ ]:
# Implementar Scaled Dot-Product Attention desde cero
# Ecuación (1) del paper

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Implementación directa de la Ecuación (1):
    Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) * V
    
    Q: (batch, seq_q, d_k)
    K: (batch, seq_k, d_k)
    V: (batch, seq_k, d_v)
    """
    d_k = Q.shape[-1]
    
    # Paso 1: QK^T / sqrt(d_k)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)
    # scores: (batch, seq_q, seq_k) — score entre cada par (q_i, k_j)
    
    # Paso 2: Softmax (los scores → pesos de atención)
    pesos = F.softmax(scores, dim=-1)
    # pesos: (batch, seq_q, seq_k) — para cada query, distribución sobre keys
    
    # Paso 3: Suma ponderada de Values
    salida = torch.matmul(pesos, V)
    # salida: (batch, seq_q, d_v)
    
    return salida, pesos


# Ejemplo concreto: 5 tokens, dim=4
seq_len = 5
d_k = 4

Q = torch.randn(1, seq_len, d_k)
K = torch.randn(1, seq_len, d_k)
V = torch.randn(1, seq_len, d_k)

salida, pesos = scaled_dot_product_attention(Q, K, V)

print("Scaled Dot-Product Attention")
print(f"Q shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")
print(f"\nPesos de atención (1 cabeza, 5 tokens):")
print(pesos[0].detach().numpy().round(3))
print("(cada fila suma 1.0 — es una distribución de probabilidad)")
print(f"\nSalida shape: {salida.shape}  (misma forma que V)")

In [ ]:
# Visualizar pesos de atención — lo que hace la herramienta BertViz

# Simular una oración: "el perro come la pelota"
tokens = ["el", "perro", "come", "la", "pelota"]
n = len(tokens)

# Crear embeddings ficticios y calcular atención
torch.manual_seed(7)
Q_orac = torch.randn(1, n, 8)
K_orac = torch.randn(1, n, 8)
V_orac = torch.randn(1, n, 8)

_, pesos_orac = scaled_dot_product_attention(Q_orac, K_orac, V_orac)
pesos_np = pesos_orac[0].detach().numpy()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(pesos_np, cmap='Blues', vmin=0, vmax=pesos_np.max())

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(tokens, fontsize=12)
ax.set_yticklabels(tokens, fontsize=12)
ax.set_xlabel('Keys (¿a qué presta atención?)', fontsize=11)
ax.set_ylabel('Queries (token que procesa)', fontsize=11)
ax.set_title('Mapa de Atención: "el perro come la pelota"\n(pesos aleatorios — solo para ilustrar)', 
             fontsize=11, fontweight='bold')

# Agregar valores
for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{pesos_np[i,j]:.2f}', ha='center', va='center', 
                fontsize=9, color='black' if pesos_np[i,j] < 0.4 else 'white')

plt.colorbar(im, ax=ax, label='Peso de atención')
plt.tight_layout()
plt.show()

print("En un modelo entrenado, estos pesos no serían aleatorios:")
print("'come' prestaría alta atención a 'perro' (sujeto) y 'pelota' (objeto)")

In [ ]:
# ¿Por qué dividir por sqrt(d_k)?
# Sección 3.2.1 del paper lo explica brevemente. Aquí la demostración.

print("¿Por qué sqrt(d_k) en el denominador?")
print("=" * 45)

d_k_vals = [4, 16, 64, 256, 512]

for d in d_k_vals:
    # Vectores aleatorios normales
    q = torch.randn(1000, d)
    k = torch.randn(1000, d)
    
    # Dot product sin escalar
    scores_sin = (q * k).sum(dim=-1)  # dot product
    # Dot product con escalar
    scores_con = scores_sin / (d ** 0.5)
    
    # Softmax
    # Cuando los scores son muy grandes, softmax se satura (gradientes pequeños)
    # Simular con batches de 10
    batch_scores_sin = torch.randn(100, d) @ torch.randn(d, 10) 
    batch_scores_con = batch_scores_sin / (d ** 0.5)
    
    std_sin = scores_sin.std().item()
    std_con = scores_con.std().item()
    
    print(f"d_k={d:>4}: std(scores) sin escalar = {std_sin:>6.2f} | con escalar = {std_con:>5.2f}")

print()
print("Sin escalar: con d_k grande, los scores tienen varianza alta")
print("→ softmax se satura (valores extremos → gradientes ≈ 0)")
print("Con sqrt(d_k): varianza normalizada a ~1.0 independiente del tamaño")
print("→ gradientes fluyen bien durante el entrenamiento")

In [ ]:
# Multi-Head Attention — Sección 3.2.2
# Idea: en lugar de una atención de d_model dimensiones,
# usar h atenciones de d_model/h dimensiones y concatenar

class MultiHeadAttention(nn.Module):
    """Multi-Head Attention de la Sección 3.2.2."""
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # dimensión por cabeza
        
        # Proyecciones lineales W^Q, W^K, W^V, W^O
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
    
    def split_heads(self, x):
        """(batch, seq, d_model) → (batch, heads, seq, d_k)"""
        batch, seq, d_model = x.shape
        x = x.view(batch, seq, self.num_heads, self.d_k)
        return x.transpose(1, 2)  # (batch, heads, seq, d_k)
    
    def forward(self, x):
        batch, seq, _ = x.shape
        
        # Proyectar y dividir en cabezas
        Q = self.split_heads(self.W_Q(x))
        K = self.split_heads(self.W_K(x))
        V = self.split_heads(self.W_V(x))
        
        # Atención en cada cabeza (en paralelo)
        attn_out, _ = scaled_dot_product_attention(Q, K, V)
        # attn_out: (batch, heads, seq, d_k)
        
        # Concatenar cabezas y proyectar
        attn_out = attn_out.transpose(1, 2).contiguous()
        attn_out = attn_out.view(batch, seq, -1)  # (batch, seq, d_model)
        
        return self.W_O(attn_out)


# Probar con dimensiones del paper: d_model=512, h=8 cabezas
mha = MultiHeadAttention(d_model=512, num_heads=8)

x_test = torch.randn(2, 10, 512)  # (batch=2, seq=10, d_model=512)
salida_mha = mha(x_test)

print("Multi-Head Attention")
print(f"Input:  {x_test.shape}")
print(f"Output: {salida_mha.shape}  ← misma forma")
print(f"\nCada cabeza trabaja en dimensión: {512//8} = 512/8")
print("8 cabezas × 64 dims = 512 dims total")
print("\nVentaja: cada cabeza puede aprender a atender distintos patrones")
print("  Cabeza 1: relaciones sujeto-verbo")
print("  Cabeza 2: relaciones de coreferencencia")
print("  Cabeza 3: dependencias de largo alcance")
print("  ...etc")

In [ ]:
# Un bloque completo del Transformer
# Combinando todo lo que hemos visto

class TransformerBlock(nn.Module):
    """
    Un bloque del Transformer (Figura 1 del paper, parte del Encoder).
    Se repite 6 veces en el Encoder y 6 en el Decoder.
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        
        # Sub-capa 1: Multi-Head Self-Attention
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)  # "Add & Norm" del diagrama
        
        # Sub-capa 2: Feed-Forward Network
        self.ffn = FeedForwardNetwork(d_model, d_ff, dropout)
        self.norm2 = nn.LayerNorm(d_model)  # "Add & Norm" del diagrama
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # Residual connection 1: x + Attention(x)
        attn_out = self.attention(x)
        x = self.norm1(x + self.dropout(attn_out))  # Add & Norm
        
        # Residual connection 2: x + FFN(x)
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))   # Add & Norm
        
        return x


# Apilar 6 bloques como en el paper
class TransformerEncoder(nn.Module):
    def __init__(self, d_model=512, num_heads=8, d_ff=2048, num_layers=6):
        super().__init__()
        self.bloques = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])
    
    def forward(self, x):
        for bloque in self.bloques:
            x = bloque(x)
        return x


# Crear encoder como en el paper
encoder = TransformerEncoder(d_model=512, num_heads=8, d_ff=2048, num_layers=6)

# Contar parámetros
total = sum(p.numel() for p in encoder.parameters())
print("Encoder del Transformer (6 bloques, como en el paper)")
print(f"Total de parámetros: {total:,}")
print()

# Forward pass
x = torch.randn(2, 10, 512)
with torch.no_grad():
    salida = encoder(x)
print(f"Input:  {x.shape}  (batch=2, seq=10, d=512)")
print(f"Output: {salida.shape}")
print()
print("Cada token pasó por 6 capas de atención + FFN.")
print("El output de cada posición contiene información de TODOS los tokens.")

### ¿Qué son las Residual Connections?

En el diagrama del paper aparece "Add & Norm". El "Add" es una **conexión residual**:

$$\text{output} = \text{LayerNorm}(x + \text{SubLayer}(x))$$

En lugar de aprender una transformación completa, la subcapa aprende **la corrección** que se debe hacer sobre la entrada. Esto resuelve el problema de gradientes que desaparecen en redes profundas.

**Intuición:** Si la subcapa no aprendió nada útil todavía, el output es $x + 0 = x$ — la información no se destruye. Conforme entrena, aprende refinamientos incrementales.

---
## Parte 5: Ejercicios de Cierre

Los siguientes ejercicios conectan directamente con secciones específicas del paper. Para cada uno se indica la sección de referencia.

In [ ]:
# ============================================================
# Ejercicio 1: Positional Encoding — Sección 3.5 del paper
# ============================================================
# El Transformer no tiene recurrencia → no sabe el orden de los tokens.
# Solución: sumar a cada embedding una codificación posicional.
#
# Fórmulas del paper:
#   PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
#   PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))

def positional_encoding(max_len, d_model):
    """
    TODO: Implementar el Positional Encoding de la Sección 3.5.
    
    Retornar un tensor de shape (max_len, d_model)
    donde PE[pos, 2i] = sin(pos / 10000^(2i/d_model))
          PE[pos, 2i+1] = cos(pos / 10000^(2i/d_model))
    """
    PE = torch.zeros(max_len, d_model)
    
    pos = torch.arange(0, max_len).unsqueeze(1).float()
    # dim_i va de 0 a d_model/2
    i = torch.arange(0, d_model, 2).float()
    
    # Calcular los divisores: 10000^(2i/d_model)
    divisor = torch.pow(10000, i / d_model)
    
    # Completar: 
    # PE[:, 0::2] = ...  # índices pares → sin
    # PE[:, 1::2] = ...  # índices impares → cos
    
    # --- TU CÓDIGO AQUÍ ---
    PE[:, 0::2] = torch.sin(pos / divisor)
    PE[:, 1::2] = torch.cos(pos / divisor)
    # ----------------------
    
    return PE


# Calcular y visualizar
PE = positional_encoding(max_len=50, d_model=64)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap del PE
im = axes[0].imshow(PE.numpy(), aspect='auto', cmap='RdYlBu')
axes[0].set_xlabel('Dimensión del embedding', fontsize=11)
axes[0].set_ylabel('Posición en la secuencia', fontsize=11)
axes[0].set_title('Positional Encoding\n(Figura del paper, Sección 3.5)', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=axes[0])

# Mostrar algunas dimensiones
for dim, color in zip([0, 1, 4, 5, 20, 21], 
                       ['#E53935','#FF7043','#FB8C00','#FDD835','#43A047','#00ACC1']):
    axes[1].plot(PE[:, dim].numpy(), label=f'dim {dim}', color=color, linewidth=1.5)

axes[1].set_xlabel('Posición', fontsize=11)
axes[1].set_ylabel('Valor del encoding', fontsize=11)
axes[1].set_title('Positional Encoding por dimensión', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=8, ncol=2)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Propiedades importantes:")
print("1. Cada posición tiene un vector único")
print("2. La similitud entre posiciones cercanas es mayor que entre lejanas")
print("3. El modelo puede generalizar a secuencias más largas que las de entrenamiento")

In [ ]:
# ============================================================
# Ejercicio 2: Contar parámetros reales del paper
# ============================================================
# El paper menciona 65M parámetros (base) y 213M (big).
# Vamos a calcularlo.

def contar_params_transformer(d_model, num_heads, d_ff, num_layers, vocab_size):
    """
    Calcular el número total de parámetros de un Transformer Encoder.
    """
    params_por_bloque = 0
    
    # Multi-Head Attention: 4 matrices (W_Q, W_K, W_V, W_O) + biases
    params_mha = 4 * (d_model * d_model + d_model)  # Wmatriz + bias
    
    # FFN: 2 capas lineales
    params_ffn = (d_model * d_ff + d_ff) + (d_ff * d_model + d_model)
    
    # LayerNorm: 2 por bloque, cada uno tiene 2*d_model params
    params_ln = 2 * 2 * d_model
    
    params_por_bloque = params_mha + params_ffn + params_ln
    
    # Total encoder
    params_encoder = num_layers * params_por_bloque
    
    # Embedding
    params_embedding = vocab_size * d_model
    
    total = params_encoder + params_embedding
    return total, params_por_bloque, params_embedding


print("Conteo de parámetros — Transformer Base (Tabla 3 del paper)")
print("=" * 60)

# Transformer Base: d_model=512, h=8, d_ff=2048, N=6
total, por_bloque, embed = contar_params_transformer(
    d_model=512, num_heads=8, d_ff=2048, num_layers=6, vocab_size=37000
)

print(f"Configuración: d_model=512, heads=8, d_ff=2048, layers=6")
print(f"  Params por bloque:  {por_bloque:>12,}")
print(f"  Params embedding:   {embed:>12,}")
print(f"  TOTAL (solo encoder): {total:>10,}")
print(f"  (el paper reporta 65M para encoder+decoder completo)")

print()
print("Comparación con modelos modernos:")
modelos = [
    ("Transformer Base (2017)", 65e6),
    ("BERT-base (2018)",        110e6),
    ("GPT-2 (2019)",            117e6),
    ("GPT-3 (2020)",            175e9),
    ("LLaMA 7B (2023)",         7e9),
    ("Mistral 7B (2023)",       7.3e9),
]
for nombre, params in modelos:
    if params >= 1e9:
        print(f"  {nombre:<30} {params/1e9:>6.1f}B parámetros")
    else:
        print(f"  {nombre:<30} {params/1e6:>6.0f}M parámetros")

In [ ]:
# ============================================================
# Ejercicio 3 (Desafío): Implementar Masked Self-Attention
# ============================================================
# En el Decoder, la atención debe ser CAUSAL: el token en posición t
# solo puede atender a las posiciones 0..t, no al futuro.
#
# Esto se implementa con un 'mask' que pone -inf en posiciones futuras
# antes del softmax.

def masked_attention(Q, K, V):
    """
    Atención causal (para el Decoder).
    El token i solo puede atender a tokens j <= i.
    """
    d_k = Q.shape[-1]
    seq_len = Q.shape[-2]
    
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)
    
    # Crear máscara causal: triangular superior = -inf
    # La posición (i,j) con j>i queda enmascarada
    mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
    scores = scores.masked_fill(mask, float('-inf'))
    
    pesos = F.softmax(scores, dim=-1)
    # Los -inf se convierten en 0 después del softmax
    
    return torch.matmul(pesos, V), pesos


# Visualizar la máscara
n = 6
Q_t = torch.randn(1, n, 8)
K_t = torch.randn(1, n, 8)
V_t = torch.randn(1, n, 8)

_, pesos_masked = masked_attention(Q_t, K_t, V_t)
pesos_m = pesos_masked[0].detach().numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pesos sin máscara
_, pesos_no_masked = scaled_dot_product_attention(Q_t, K_t, V_t)
axes[0].imshow(pesos_no_masked[0].detach().numpy(), cmap='Blues')
axes[0].set_title('Atención Normal\n(Encoder: ve todo)', fontweight='bold')
axes[0].set_xlabel('Posición Key')
axes[0].set_ylabel('Posición Query')

# Pesos con máscara
axes[1].imshow(pesos_m, cmap='Blues')
axes[1].set_title('Masked Self-Attention\n(Decoder: solo ve el pasado)', fontweight='bold')
axes[1].set_xlabel('Posición Key')
axes[1].set_ylabel('Posición Query')

for ax in axes:
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))

plt.tight_layout()
plt.show()

print("El Decoder usa masked attention para evitar 'mirar al futuro'")
print("durante el entrenamiento (el modelo no puede hacer trampa)")
print("\nConexión con el paper: Sección 3.1, párrafo del Decoder")

---
## Resumen: El Mapa Completo

Todo lo que implementamos hoy está en el paper. Aquí el mapa:

| Componente | Sección del Paper | Qué es |
|:-----------|:-----------------|:-------|
| `nn.Linear` | 3.3 | Transformación lineal $xW^T + b$ |
| `F.relu` / `F.gelu` | 3.3 | Función de activación |
| `FeedForwardNetwork` | 3.3 | Dos capas lineales con ReLU |
| `scaled_dot_product_attention` | 3.2.1 | Ecuación (1) |
| `MultiHeadAttention` | 3.2.2 | Ecuación (2) |
| `nn.LayerNorm` | 3.1 | Normalización (Add & Norm) |
| `TransformerBlock` | Figura 1 | Un bloque del encoder |
| `positional_encoding` | 3.5 | Ecuaciones (3) y (4) |
| `masked_attention` | 3.2.3 | Decoder auto-regresivo |

### Lo que viene

Con estos bloques, los modelos como BERT y GPT son variantes:
- **BERT:** Solo encoder, entrenado con Masked LM
- **GPT:** Solo decoder, entrenado para predecir el siguiente token  
- **T5:** Encoder-decoder, todo texto a texto
- **LLaMA/Mistral:** Decoder mejorado con RoPE, GQA y otras optimizaciones

---
**Próxima sesión:** Fine-tuning — cómo adaptar estos modelos preentrenados a tu tarea específica.

In [ ]:
# Pregunta de reflexión final
# (Responder en Google Classroom como tarea breve)

print("="*60)
print("TAREA PARA LA PRÓXIMA SESIÓN")
print("="*60)
print()
print("Vuelve a leer la Sección 3 de 'Attention Is All You Need'")
print("con el notebook como referencia. Responde en Classroom:")
print()
print("1. El paper dice que usan h=8 cabezas con d_model=512.")
print("   ¿Cuánto es d_k por cabeza? ¿Por qué esa elección?")
print()
print("2. ¿Cuál es la diferencia entre self-attention en el encoder")
print("   y en el decoder? ¿Por qué existe esa diferencia?")
print()
print("3. El FFN usa dim_ff=2048 = 4 × d_model.")
print("   ¿Qué pasa si usas 2× o 8×? ¿Trade-off?")
print()
print("(No hay respuesta única — se evalúa razonamiento)")